# The Final RAG Pipeline: Grounding AI in Private Data
We are moving from "Chatting with an AI" to "Building a Knowledge Engine." 

**The RAG Workflow:**
1. **Ingest:** Load and chunk our private data (*Hamlet*).
2. **Index:** Convert those chunks into embeddings (vectors).
3. **Retrieve:** When a user asks a question, find the most relevant chunks.
4. **Augment:** Stuff those chunks into the LLM's prompt as "Context."
5. **Generate:** The LLM answers the question based *only* on the provided context.

In [3]:
#import os
import numpy as np
import gradio as gr
from openai import OpenAI
from litellm import completion
#from dotenv import load_dotenv
#from Warning import warnings
#warnings.filterwarnings('ignore')

#load_dotenv(override=True)
client = OpenAI(
        base_url="http://localhost:11434/v1",
)

print("🏗️ RAG Pipeline Environment Initialized!")

🏗️ RAG Pipeline Environment Initialized!


## Part 1: Preparing the Knowledge Base(NA given pdf as base)

We load the text, break it into 1,000-character chunks, and convert them into vectors. In a real system, this "Indexing" phase would happen once and be stored in a database.

In [4]:
"""
# 1. Load Document
with open("hamlet.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 2. Simple Recursive Chunking
chunk_size = 1000
overlap = 150
chunks = []
for i in range(0, len(text), chunk_size - overlap):
    chunks.append(text[i:i + chunk_size])

print(f"📊 Created {len(chunks)} chunks.")

# 3. Vectorize (Indexing)
def get_embedding(text):
    return client.embeddings.create(input=[text], model="qwen3-embedding").data[0].embedding

print("🧠 Embedding knowledge base... (this takes a few seconds)")
chunk_vectors = [get_embedding(c) for c in chunks]
print("✅ Indexing Complete.")
"""

'\n# 1. Load Document\nwith open("hamlet.txt", "r", encoding="utf-8") as f:\n    text = f.read()\n\n# 2. Simple Recursive Chunking\nchunk_size = 1000\noverlap = 150\nchunks = []\nfor i in range(0, len(text), chunk_size - overlap):\n    chunks.append(text[i:i + chunk_size])\n\nprint(f"📊 Created {len(chunks)} chunks.")\n\n# 3. Vectorize (Indexing)\ndef get_embedding(text):\n    return client.embeddings.create(input=[text], model="qwen3-embedding").data[0].embedding\n\nprint("🧠 Embedding knowledge base... (this takes a few seconds)")\nchunk_vectors = [get_embedding(c) for c in chunks]\nprint("✅ Indexing Complete.")\n'

## Part 1.1 PDF reader sample 

In [21]:
from pypdf import PdfReader
reader = PdfReader("00_EXL_OFFER_LETTER.pdf")
text = "\n".join(page.extract_text() or "" for page in reader.pages)
#print(text)
# 2. Simple Recursive Chunking
chunk_size = 1000
overlap = 150
chunks = []
for i in range(0, len(text), chunk_size - overlap):
    chunks.append(text[i:i + chunk_size])

print(f"📊 Created {len(chunks)} chunks.")

# 3. Vectorize (Indexing)
def get_embedding(text):
    return client.embeddings.create(input=[text], model="qwen3-embedding").data[0].embedding

print("🧠 Embedding knowledge base... (this takes a few seconds)")
chunk_vectors = [get_embedding(c) for c in chunks]
print("✅ Indexing Complete.")

📊 Created 85 chunks.
🧠 Embedding knowledge base... (this takes a few seconds)
✅ Indexing Complete.


## Part 2: The Retrieval Engine

We use Cosine Similarity to find the 3 most relevant "Knowledge Nuggets" for any user query.

In [22]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def get_relevant_context(query, top_k=5):
    query_vec = get_embedding(query)
    scores = [cosine_similarity(query_vec, cv) for cv in chunk_vectors]
    
    # Get indices of top_k highest scores
    top_indices = np.argsort(scores)[-top_k:][::-1]
    
    # DIAGNOSTIC: Print top score
    print(f"🔍 Retrieval Match: Top Score = {scores[top_indices[0]]:.4f}")
    
    return [chunks[i] for i in top_indices]

## Part 3: The RAG Brain (Streaming & Grounding)

We construct a prompt that forces the AI to use the retrieved context. We then stream the result to the UI.

In [23]:
def rag_chat(message, history):
    # 1. Retrieve the Facts
    relevant_chunks = get_relevant_context(message)
    context_block = "\n---\n".join(relevant_chunks)
    print(context_block)
    # 2. Build the Grounded Prompt
    system_prompt = f"""You are a HR and having a offer letter about an new joinee employee. 
    Explain him about his offer from the provided data,
    Make Sure you give him response with accurate amounts if anything applicable
    Make The reposne max to 5 lines and provide in bulleted points always to have a short and sweet response'    
    
    CONTEXT:
    {context_block}
    """
    
    messages = [{"role": "system", "content": system_prompt}]
    
    # 3. Add History (Robust handling for all Gradio versions)
    for msg in history:
        if hasattr(msg, 'role'):
            messages.append({"role": msg.role, "content": msg.content})
        elif isinstance(msg, dict):
            messages.append({"role": msg['role'], "content": msg['content']})
        elif isinstance(msg, (list, tuple)) and len(msg) == 2:
            messages.append({"role": "user", "content": msg[0]})
            messages.append({"role": "assistant", "content": msg[1]})
    
    messages.append({"role": "user", "content": message})
    
    # 4. Stream the grounded response
    response = completion(
        model="ollama/mistral",
         messages=messages,
         stream=True
         )
    
    partial_text = ""
    for chunk in response:
        token = chunk.choices[0].delta.content
        if token:
            partial_text += token
            yield partial_text

## Part 4: The Capstone UI

We wrap our RAG logic in a `gr.ChatInterface` to create a professional, grounded application.

In [24]:
demo = gr.ChatInterface(
    fn=rag_chat,
    title="EXL Offer Letter",
    description="Ask me anything about the Offer Letter",
    examples=["What is the document is all about", "Explain about EXL policies and culture"]
)

print("🚀 RAG Pipeline Wired and Ready.")
demo.launch()

🚀 RAG Pipeline Wired and Ready.
* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


/Users/bharathraskal/Documents/Raskal/Professional/Repo/llm-rag/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


🔍 Retrieval Match: Top Score = 0.5623
, approved and paid during the March/April timeframe of each year as determined by the Board in its sole
Page 5 of 21
 
SCIOinspire Consulting Services (India) Private Limited 
Registered Office: TVH Beliciaa Towers, Tower 1,6th Floor, Block No.94, MRC Nagar, Chennai, Tamilnadu 600028 
Phone: 044 30994800 
 
 
 
 
 
 
 
 
 
discretion.
 
6.1 As compensation for services to be rendered, you shall be paid a Basic Salary of Rs. 6,30,000  per annum. The salary
shall be payable on monthly basis in arrears on or about the last working day of each calendar month but before expiry of
the 7th day of the succeeding calendar month.  Other allowances and benefits payable shall be as detailed in Appendix 1
hereto.
 
6.2 The payment of all compensation shall be made in accordance with the relevant policies of the Company in effect from
time to time, including normal payroll practices, and shall be subject to income tax deductions at source, as applicable. All
re

/Users/bharathraskal/Documents/Raskal/Professional/Repo/llm-rag/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/bharathraskal/Documents/Raskal/Professional/Repo/llm-rag/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


🔍 Retrieval Match: Top Score = 0.5965
, approved and paid during the March/April timeframe of each year as determined by the Board in its sole
Page 5 of 21
 
SCIOinspire Consulting Services (India) Private Limited 
Registered Office: TVH Beliciaa Towers, Tower 1,6th Floor, Block No.94, MRC Nagar, Chennai, Tamilnadu 600028 
Phone: 044 30994800 
 
 
 
 
 
 
 
 
 
discretion.
 
6.1 As compensation for services to be rendered, you shall be paid a Basic Salary of Rs. 6,30,000  per annum. The salary
shall be payable on monthly basis in arrears on or about the last working day of each calendar month but before expiry of
the 7th day of the succeeding calendar month.  Other allowances and benefits payable shall be as detailed in Appendix 1
hereto.
 
6.2 The payment of all compensation shall be made in accordance with the relevant policies of the Company in effect from
time to time, including normal payroll practices, and shall be subject to income tax deductions at source, as applicable. All
re

/Users/bharathraskal/Documents/Raskal/Professional/Repo/llm-rag/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
